# 01 — Build the county-month analysis panel

**Study:** Wildfire-associated environmental disturbance and H5N1 detection risk in California (2022–2025).
**Unit of analysis:** county-month (58 counties × 48 months).

This notebook assembles the analysis panel:
1. Outcome — ≥1 confirmed H5N1 wild-bird detection in the county-month (USDA APHIS).
2. Disturbance/exposure — 1-month-lagged FIRMS active fire (count, FRP), MODIS NDVI and LST anomalies, PRISM climate, CDL agricultural share.

Out of the box it runs on the built-in **synthetic panel** (`USE_SYNTHETIC = True` in `src/config.py`) so the full pipeline is reproducible with zero downloads. With real data in `data/raw/` (see `data/README.md`), set `USE_SYNTHETIC = False` and rerun — the downstream notebooks are unchanged.

Key design choice: **all disturbance predictors enter at lag 1 month**, matching the conceptual sequence *disturbance → habitat change → waterfowl redistribution → detection*.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # run from notebooks/

import pandas as pd
from src import config
from src.features import add_seasonality, add_transforms, save_panel

## 1. Load outcome + environmental layers

In [ ]:
if config.USE_SYNTHETIC:
    from src.synthetic import make_panel
    panel = make_panel(seed=config.RANDOM_STATE)
    panel["_synthetic"] = True
    print("Synthetic panel:", panel.shape)
else:
    raise SystemExit(
        "Set up real data per data/README.md, then build the panel here "
        "(APHIS outcome + FIRMS/MODIS/PRISM/CDL county-month aggregates)."
    )

## 2. Feature engineering: seasonality and lagged disturbance

In [ ]:
panel = add_seasonality(panel)
panel = add_transforms(panel)
save_panel(panel)
panel.head(3)

## 3. Sanity checks

In [ ]:
print("Counties:", panel['county'].nunique(), "| Months:", panel['month'].nunique())
print("Prevalence (county-months with >=1 detection): {:.1%}".format(panel['y'].mean()))
panel[['y','fire_count','ndvi_anom','precip','tmean']].describe().round(2)